# Generate Location Split Dataset

This notebook generates patient bundles split between two hospitals using a **location strategy**.
Encounters are grouped by location, and different locations are partitioned between hospitals.

## Import Libraries and Load Resources

Load the resources, graphs, and helper functions from the main pipeline.

In [ ]:
from pathlib import Path
from strategies import SPLIT_STRATEGIES
from dataset_factory import build_split_dataset
from persistence import persist_patient_bundles

## Build Graphs and Load Resources

Load the FHIR resources from compressed NDJSON files and build forward/reverse reference graphs.

In [ ]:
from graph_builder import build_fhir_graphs

# Load FHIR resources and build graphs
forward_graph, reverse_graph, resources_map = build_fhir_graphs()

## Configure Strategy

Select the location splitting strategy.

In [ ]:
# Select strategy and dataset name
strategy = SPLIT_STRATEGIES["location"]
dataset_name = "location"
seed = 42  # Optional: set seed for reproducibility

print(f"Strategy: {strategy.__name__}")
print(f"Dataset name: {dataset_name}")

## Build Split Dataset

Use the strategy to split patient encounters between hospital_A and hospital_B.

In [ ]:
print("Building split dataset...")
patient_bundles_a, patient_bundles_b = build_split_dataset(
    strategy=strategy,
    resources_map=resources_map,
    reverse_graph=reverse_graph,
    forward_graph=forward_graph,
    seed=seed
)

print(f"Created {len(patient_bundles_a)} bundles for hospital_A")
print(f"Created {len(patient_bundles_b)} bundles for hospital_B")
print(f"Patients with split encounters: {len(patient_bundles_b)}")

## Persist Datasets to Disk

Save the split bundles to `input/{dataset_name}/hospital_A` and `input/{dataset_name}/hospital_B`.

In [ ]:
print("Persisting bundles...")

persisted_a = persist_patient_bundles(
    patient_bundles_a,
    Path(f"input/{dataset_name}/hospital_A"),
    resources_map
)

persisted_b = persist_patient_bundles(
    patient_bundles_b,
    Path(f"input/{dataset_name}/hospital_B"),
    resources_map
)

print(f"Saved {persisted_a} bundles to input/{dataset_name}/hospital_A")
print(f"Saved {persisted_b} bundles to input/{dataset_name}/hospital_B")
print(f"\nTotal bundles: {persisted_a + persisted_b}")